In [ ]:
!rm -rf /content/ecommerce-product-analytics

In [ ]:
!git clone https://github.com/aleksvoronova/ecommerce-product-analytics.git /content/ecommerce-product-analytics

In [ ]:
# 0. НАСТРОЙКА ПРОЕКТА


from google.colab import drive
from pathlib import Path

import sys
import pandas as pd
import numpy as np



# Google Drive


drive.mount("/content/drive")



# Пути к данным


DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/ecommerce-product-analytics"
)

RAW_DIR = DRIVE_PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = DRIVE_PROJECT_DIR / "data" / "processed"
DOCS_DIR = DRIVE_PROJECT_DIR / "docs"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCS_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# GitHub repository


REPO_DIR = Path(
    "/content/ecommerce-product-analytics"
)

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("DOCS_DIR:", DOCS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAW_DIR: /content/drive/MyDrive/ecommerce-product-analytics/data/raw
PROCESSED_DIR: /content/drive/MyDrive/ecommerce-product-analytics/data/processed
DOCS_DIR: /content/drive/MyDrive/ecommerce-product-analytics/docs


In [ ]:

# 2. SRC status


!ls -la /content/ecommerce-product-analytics/src

total 20
drwxr-xr-x 2 root root 4096 Sep 17 07:06 .
drwxr-xr-x 6 root root 4096 Sep 17 07:06 ..
-rw-r--r-- 1 root root 8084 Sep 17 07:06 clean_data.py
-rw-r--r-- 1 root root    0 Sep 17 07:06 __init__.py
-rw-r--r-- 1 root root 2214 Sep 17 07:06 load_data.py


In [ ]:
# 3. Importing project's modules


if str(REPO_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(REPO_DIR)
    )


from src.load_data import load_raw_data

from src.clean_data import (
    clean_customers,
    clean_orders,
    clean_order_items,
    clean_payments,
    clean_reviews,
    clean_products,
    clean_sellers,
    clean_geolocation,
)


print(" Модули проекта успешно импортированы")

✓ Модули проекта успешно импортированы


In [ ]:
# 4. ЗАГРУЗКА RAW DATA

dataframes = load_raw_data(
    RAW_DIR
)


# Удобные переменные

customers = dataframes[
    "olist_customers_dataset"
]

orders = dataframes[
    "olist_orders_dataset"
]

items = dataframes[
    "olist_order_items_dataset"
]

payments = dataframes[
    "olist_order_payments_dataset"
]

reviews = dataframes[
    "olist_order_reviews_dataset"
]

products = dataframes[
    "olist_products_dataset"
]

sellers = dataframes[
    "olist_sellers_dataset"
]

categories = dataframes[
    "product_category_name_translation"
]


if "olist_geolocation_dataset" in dataframes:
    geolocation = dataframes[
        "olist_geolocation_dataset"
    ]
else:
    geolocation = None


print("\n Исходные таблицы загружены")

ЗАГРУЗКА RAW DATA
✓ olist_customers_dataset                 99,441 строк ×   5 столбцов
✓ olist_geolocation_dataset            1,000,163 строк ×   5 столбцов
✓ olist_order_items_dataset              112,650 строк ×   7 столбцов
✓ olist_order_payments_dataset           103,886 строк ×   5 столбцов
✓ olist_order_reviews_dataset             99,224 строк ×   7 столбцов
✓ olist_orders_dataset                    99,441 строк ×   8 столбцов
✓ olist_products_dataset                  32,951 строк ×   9 столбцов
✓ olist_sellers_dataset                    3,095 строк ×   4 столбцов
✓ product_category_name_translation           71 строк ×   2 столбцов
RAW DATA УСПЕШНО ЗАГРУЖЕНЫ

✓ Исходные таблицы загружены


In [ ]:
# 5. СОСТОЯНИЕ ДАННЫХ ДО ОЧИСТКИ


raw_tables = {
    "customers": customers,
    "orders": orders,
    "order_items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
}

if geolocation is not None:
    raw_tables["geolocation"] = geolocation


before_summary = []

for name, df in raw_tables.items():

    before_summary.append({
        "table": name,
        "rows_before": len(df),
        "columns_before": len(df.columns),

        "full_duplicates_before":
            int(df.duplicated().sum()),

        "missing_cells_before":
            int(df.isna().sum().sum())
    })


before_summary = pd.DataFrame(
    before_summary
)

display(before_summary)

,table,rows_before,columns_before,full_duplicates_before,missing_cells_before
0,customers,99441,5,0,0
1,orders,99441,8,0,4908
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,0,145903
5,products,32951,9,0,2448
6,sellers,3095,4,0,0
7,geolocation,1000163,5,261831,0


In [ ]:
# 6. CLEANING PIPELINE

customers_clean = clean_customers(
    customers
)

orders_clean = clean_orders(
    orders
)

items_clean = clean_order_items(
    items
)

payments_clean = clean_payments(
    payments
)

reviews_clean = clean_reviews(
    reviews
)

products_clean = clean_products(
    products,
    categories
)

sellers_clean = clean_sellers(
    sellers
)


if geolocation is not None:

    geolocation_clean = clean_geolocation(
        geolocation
    )

else:

    geolocation_clean = None


print(" Cleaning pipeline выполнен")

✓ Cleaning pipeline выполнен


In [ ]:
# 7. ПРОВЕРКА УНИКАЛЬНОСТИ КЛЮЧЕЙ

key_checks = pd.DataFrame({

    "table": [
        "customers",
        "orders",
        "order_items",
        "products",
        "sellers"
    ],

    "key": [
        "customer_id",
        "order_id",
        "order_id + order_item_id",
        "product_id",
        "seller_id"
    ],

    "duplicate_keys": [

        customers_clean[
            "customer_id"
        ].duplicated().sum(),

        orders_clean[
            "order_id"
        ].duplicated().sum(),

        items_clean.duplicated(
            subset=[
                "order_id",
                "order_item_id"
            ]
        ).sum(),

        products_clean[
            "product_id"
        ].duplicated().sum(),

        sellers_clean[
            "seller_id"
        ].duplicated().sum()
    ]
})


display(key_checks)

,table,key,duplicate_keys
0,customers,customer_id,0
1,orders,order_id,0
2,order_items,order_id + order_item_id,0
3,products,product_id,0
4,sellers,seller_id,0


In [ ]:
# 8. ПРОВЕРКА ЧИСЛОВЫХ АНОМАЛИЙ


numeric_anomalies = pd.DataFrame({

    "check": [
        "price < 0",
        "freight_value < 0",
        "payment_value < 0",
        "payment_installments < 0"
    ],

    "count": [

        int(
            (items_clean["price"] < 0).sum()
        ),

        int(
            (
                items_clean["freight_value"] < 0
            ).sum()
        ),

        int(
            (
                payments_clean["payment_value"] < 0
            ).sum()
        ),

        int(
            (
                payments_clean[
                    "payment_installments"
                ] < 0
            ).sum()
        )
    ]
})


display(numeric_anomalies)

,check,count
0,price < 0,0
1,freight_value < 0,0
2,payment_value < 0,0
3,payment_installments < 0,0


In [ ]:
# 9. ДИАПАЗОНЫ ЧИСЛОВЫХ ЗНАЧЕНИЙ


print("ORDER ITEMS")
display(
    items_clean[
        [
            "price",
            "freight_value",
            "item_total"
        ]
    ].describe()
)


print("\nPAYMENTS")
display(
    payments_clean[
        [
            "payment_value",
            "payment_installments"
        ]
    ].describe()
)

ORDER ITEMS


,price,freight_value,item_total
count,112650.000000,112650.000000,112650.000000
mean,120.653739,19.990320,140.644059
std,183.633928,15.806405,190.724394
min,0.850000,0.000000,6.080000
25%,39.900000,13.080000,55.220000
50%,74.990000,16.260000,92.320000
75%,134.900000,21.150000,157.937500
max,6735.000000,409.680000,6929.310000



PAYMENTS


,payment_value,payment_installments
count,103886.000000,103886.000000
mean,154.100380,2.853349
std,217.494064,2.687051
min,0.000000,0.000000
25%,56.790000,1.000000
50%,100.000000,1.000000
75%,171.837500,4.000000
max,13664.080000,24.000000


In [ ]:
# 10. ПРОВЕРКА ВРЕМЕННЫХ АНОМАЛИЙ


delivery_before_purchase = orders_clean[
    orders_clean["order_delivered_customer_date"]
    <
    orders_clean["order_purchase_timestamp"]
]


approval_before_purchase = orders_clean[
    orders_clean["order_approved_at"]
    <
    orders_clean["order_purchase_timestamp"]
]


carrier_before_approval = orders_clean[
    orders_clean["order_delivered_carrier_date"]
    <
    orders_clean["order_approved_at"]
]


estimated_before_purchase = orders_clean[
    orders_clean["order_estimated_delivery_date"]
    <
    orders_clean["order_purchase_timestamp"]
]


date_anomalies = pd.DataFrame({

    "check": [
        "Delivery before purchase",
        "Approval before purchase",
        "Carrier handoff before approval",
        "Estimated delivery before purchase"
    ],

    "count": [
        len(delivery_before_purchase),
        len(approval_before_purchase),
        len(carrier_before_approval),
        len(estimated_before_purchase)
    ]
})


display(date_anomalies)

,check,count
0,Delivery before purchase,0
1,Approval before purchase,0
2,Carrier handoff before approval,1359
3,Estimated delivery before purchase,0


In [ ]:
# 11. ПРОПУСКИ В ДАТАХ ЗАКАЗОВ ПО СТАТУСАМ

order_date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]


missing_dates_by_status = []

for status, group in orders_clean.groupby(
    "order_status",
    dropna=False
):

    row = {
        "order_status": status,
        "orders": len(group)
    }

    for col in order_date_columns:

        row[f"{col}_missing"] = int(
            group[col].isna().sum()
        )

    missing_dates_by_status.append(
        row
    )


missing_dates_by_status = pd.DataFrame(
    missing_dates_by_status
)


display(missing_dates_by_status)

,order_status,orders,order_approved_at_missing,order_delivered_carrier_date_missing,order_delivered_customer_date_missing,order_estimated_delivery_date_missing
0,approved,2,0,2,2,0
1,canceled,625,141,550,619,0
2,created,5,5,5,5,0
3,delivered,96478,14,2,8,0
4,invoiced,314,0,314,314,0
5,processing,301,0,301,301,0
6,shipped,1107,0,0,1107,0
7,unavailable,609,0,609,609,0


In [ ]:
# 12. ПРОВЕРКА REVIEWS


print("Распределение review_score:")

display(
    reviews_clean[
        "review_score"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .rename("count")
    .to_frame()
)


review_stats = pd.DataFrame({

    "metric": [
        "Reviews total",
        "Reviews with title",
        "Reviews with comment",
        "Title share (%)",
        "Comment share (%)"
    ],

    "value": [
        len(reviews_clean),

        int(
            reviews_clean[
                "has_review_title"
            ].sum()
        ),

        int(
            reviews_clean[
                "has_review_comment"
            ].sum()
        ),

        round(
            reviews_clean[
                "has_review_title"
            ].mean() * 100,
            2
        ),

        round(
            reviews_clean[
                "has_review_comment"
            ].mean() * 100,
            2
        )
    ]
})


display(review_stats)

Распределение review_score:


,count
review_score,
1,11424
2,3151
3,8179
4,19142
5,57328


,metric,value
0,Reviews total,99224.00
1,Reviews with title,11568.00
2,Reviews with comment,40977.00
3,Title share (%),11.66
4,Comment share (%),41.30


In [ ]:
# 13. GEOLOCATION


if geolocation_clean is not None:

    geolocation_summary = pd.DataFrame({

        "metric": [
            "Rows before",
            "Rows after",
            "Rows removed",
            "Full duplicates remaining"
        ],

        "value": [
            len(geolocation),

            len(geolocation_clean),

            len(geolocation)
            - len(geolocation_clean),

            int(
                geolocation_clean
                .duplicated()
                .sum()
            )
        ]
    })

    display(
        geolocation_summary
    )

else:

    print(
        "Geolocation dataset отсутствует."
    )

,metric,value
0,Rows before,1000163
1,Rows after,738332
2,Rows removed,261831
3,Full duplicates remaining,0


In [ ]:
# 14. ПРОВЕРКА ЦЕЛОСТНОСТИ СВЯЗЕЙ


missing_customers = (
    set(
        orders_clean["customer_id"]
    )
    -
    set(
        customers_clean["customer_id"]
    )
)


missing_orders_items = (
    set(
        items_clean["order_id"]
    )
    -
    set(
        orders_clean["order_id"]
    )
)


missing_orders_payments = (
    set(
        payments_clean["order_id"]
    )
    -
    set(
        orders_clean["order_id"]
    )
)


missing_orders_reviews = (
    set(
        reviews_clean["order_id"]
    )
    -
    set(
        orders_clean["order_id"]
    )
)


missing_products = (
    set(
        items_clean["product_id"]
    )
    -
    set(
        products_clean["product_id"]
    )
)


missing_sellers = (
    set(
        items_clean["seller_id"]
    )
    -
    set(
        sellers_clean["seller_id"]
    )
)


integrity_results = pd.DataFrame({

    "relationship": [
        "Orders → Customers",
        "Order Items → Orders",
        "Payments → Orders",
        "Reviews → Orders",
        "Order Items → Products",
        "Order Items → Sellers"
    ],

    "missing_keys": [
        len(missing_customers),
        len(missing_orders_items),
        len(missing_orders_payments),
        len(missing_orders_reviews),
        len(missing_products),
        len(missing_sellers)
    ]
})


display(
    integrity_results
)

NameError: name 'orders_clean' is not defined

In [ ]:
# 15. СОСТОЯНИЕ ПОСЛЕ CLEANING


clean_tables = {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
}

if geolocation_clean is not None:
    clean_tables[
        "geolocation"
    ] = geolocation_clean


after_summary = []

for name, df in clean_tables.items():

    after_summary.append({

        "table": name,

        "rows_after":
            len(df),

        "columns_after":
            len(df.columns),

        "full_duplicates_after":
            int(
                df.duplicated().sum()
            ),

        "missing_cells_after":
            int(
                df.isna().sum().sum()
            )
    })


after_summary = pd.DataFrame(
    after_summary
)


cleaning_summary = (
    before_summary
    .merge(
        after_summary,
        on="table",
        how="outer"
    )
)


cleaning_summary[
    "rows_removed"
] = (
    cleaning_summary[
        "rows_before"
    ]
    -
    cleaning_summary[
        "rows_after"
    ]
)


display(
    cleaning_summary
)

,table,rows_before,columns_before,full_duplicates_before,missing_cells_before,rows_after,columns_after,full_duplicates_after,missing_cells_after,rows_removed
0,customers,99441,5,0,0,99441,5,0,0,0
1,geolocation,1000163,5,261831,0,738332,5,0,0,261831
2,order_items,112650,7,0,0,112650,8,0,0,0
3,orders,99441,8,0,4908,99441,16,0,13803,0
4,payments,103886,5,0,0,103886,5,0,0,0
5,products,32951,9,0,2448,32951,10,0,1838,0
6,reviews,99224,7,0,145903,99224,9,0,145903,0
7,sellers,3095,4,0,0,3095,4,0,0,0


In [ ]:
# 16. ПРОВЕРКА СОЗДАННЫХ АНАЛИТИЧЕСКИХ ПРИЗНАКОВ


order_features = [
    "purchase_date",
    "purchase_month",
    "purchase_year",
    "purchase_month_num",
    "purchase_weekday",
    "delivery_days",
    "delay_days",
    "delivered_on_time"
]


display(
    orders_clean[
        [
            "order_id",
            "order_status"
        ]
        + order_features
    ].head(10)
)

,order_id,order_status,purchase_date,purchase_month,purchase_year,purchase_month_num,purchase_weekday,delivery_days,delay_days,delivered_on_time
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02,2017-10,2017,10,Monday,8.436574,-7.107488,True
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24,2018-07,2018,7,Tuesday,13.782037,-5.355729,True
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08,2018-08,2018,8,Wednesday,9.394213,-17.245498,True
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18,2017-11,2017,11,Saturday,13.208750,-12.980069,True
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13,2018-02,2018,2,Tuesday,2.873877,-9.238171,True
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-07-09,2017-07,2017,7,Sunday,16.542245,-5.543113,True
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,2017-04-11,2017-04,2017,4,Tuesday,NaN,NaN,<NA>
7,6514b8ad8028c9f2cc2374ded245783f,delivered,2017-05-16,2017-05,2017,5,Tuesday,9.989826,-11.461215,True
8,76c6e866289321a7c93b82b54852dc33,delivered,2017-01-23,2017-01,2017,1,Monday,9.818762,-31.410995,True
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,2017-07-29,2017-07,2017,7,Saturday,18.221852,-6.281597,True


In [ ]:
# 17. DELIVERY METRICS CHECK

delivery_check = pd.DataFrame({

    "metric": [
        "Orders total",
        "Orders with known delivery time",
        "Delivered on time",
        "Delivered late"
    ],

    "value": [

        len(
            orders_clean
        ),

        int(
            orders_clean[
                "delivery_days"
            ].notna().sum()
        ),

        int(
            (
                orders_clean[
                    "delivered_on_time"
                ] == True
            ).sum()
        ),

        int(
            (
                orders_clean[
                    "delivered_on_time"
                ] == False
            ).sum()
        )
    ]
})


display(
    delivery_check
)

,metric,value
0,Orders total,99441
1,Orders with known delivery time,96476
2,Delivered on time,88649
3,Delivered late,7827


In [ ]:
# 18. ФИНАЛЬНЫЕ ПРОВЕРКИ


critical_checks = {
    "customers.customer_id unique":
        customers_clean[
            "customer_id"
        ].duplicated().sum() == 0,

    "orders.order_id unique":
        orders_clean[
            "order_id"
        ].duplicated().sum() == 0,

    "products.product_id unique":
        products_clean[
            "product_id"
        ].duplicated().sum() == 0,

    "sellers.seller_id unique":
        sellers_clean[
            "seller_id"
        ].duplicated().sum() == 0,

    "no negative price":
        (
            items_clean[
                "price"
            ] < 0
        ).sum() == 0,

    "no negative freight":
        (
            items_clean[
                "freight_value"
            ] < 0
        ).sum() == 0,

    "no negative payments":
        (
            payments_clean[
                "payment_value"
            ] < 0
        ).sum() == 0
}


for check, result in critical_checks.items():

    symbol = "✓" if result else "⚠"

    print(
        f"{symbol} {check}: {result}"
    )

✓ customers.customer_id unique: True
✓ orders.order_id unique: True
✓ products.product_id unique: True
✓ sellers.seller_id unique: True
✓ no negative price: True
✓ no negative freight: True
✓ no negative payments: True


In [ ]:
# 19. СОХРАНЕНИЕ PROCESSED DATA

customers_clean.to_csv(
    PROCESSED_DIR / "customers.csv",
    index=False
)

orders_clean.to_csv(
    PROCESSED_DIR / "orders.csv",
    index=False
)

items_clean.to_csv(
    PROCESSED_DIR / "order_items.csv",
    index=False
)

payments_clean.to_csv(
    PROCESSED_DIR / "payments.csv",
    index=False
)

reviews_clean.to_csv(
    PROCESSED_DIR / "reviews.csv",
    index=False
)

products_clean.to_csv(
    PROCESSED_DIR / "products.csv",
    index=False
)

sellers_clean.to_csv(
    PROCESSED_DIR / "sellers.csv",
    index=False
)


if geolocation_clean is not None:

    geolocation_clean.to_csv(
        PROCESSED_DIR / "geolocation.csv",
        index=False
    )


print(
    "✓ Очищенные данные сохранены:"
)

print(
    PROCESSED_DIR
)

✓ Очищенные данные сохранены:
/content/drive/MyDrive/ecommerce-product-analytics/data/processed


In [ ]:
# 20. СОХРАНЕНИЕ CLEANING REPORTS

cleaning_summary.to_csv(
    DOCS_DIR / "cleaning_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


integrity_results.to_csv(
    DOCS_DIR / "referential_integrity.csv",
    index=False,
    encoding="utf-8-sig"
)


numeric_anomalies.to_csv(
    DOCS_DIR / "numeric_anomalies.csv",
    index=False,
    encoding="utf-8-sig"
)


date_anomalies.to_csv(
    DOCS_DIR / "date_anomalies.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "✓ cleaning_summary.csv"
)

print(
    "✓ referential_integrity.csv"
)

print(
    "✓ numeric_anomalies.csv"
)

print(
    "✓ date_anomalies.csv"
)

✓ cleaning_summary.csv
✓ referential_integrity.csv
✓ numeric_anomalies.csv
✓ date_anomalies.csv


In [ ]:
1. ПРОВЕРКА СОХРАНЕННЫХ ФАЙЛОВ

print("PROCESSED DATA:")


for file in sorted(
    PROCESSED_DIR.glob("*.csv")
):
    print(
        f"{file.name:<25} "
        f"{file.stat().st_size / 1024 / 1024:.2f} MB"
    )


print("\nREPORTS:")


for file in sorted(
    DOCS_DIR.glob("*.csv")
):
    print(
        file.name
    )

PROCESSED DATA:
--------------------------------------------------
customers.csv             8.17 MB
geolocation.csv           41.33 MB
order_items.csv           15.16 MB
orders.csv                22.90 MB
payments.csv              5.37 MB
products.csv              3.11 MB
reviews.csv               14.39 MB
sellers.csv               0.16 MB

REPORTS:
--------------------------------------------------
cleaning_summary.csv
date_anomalies.csv
numeric_anomalies.csv
referential_integrity.csv
